# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")


## 2. Data Overview
Review available record sets, their IDs, and the fields available in each. 

**Note:** All `@id`s are the unique references to each entity.

In [ ]:
# List all record sets and the field ids for each
record_sets = dataset.metadata.record_sets
print("Available Record Sets and their fields:")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}, name: {rs['name']}")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"    Field @id: {field['@id']} ({field.get('name', '')})")
    else:
        print("    No fields detected in this record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All Croissant entities are referenced by their `@id` field.


In [ ]:
# Example: extract all record sets into DataFrames by their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set: {record_set_id} with {len(dataframes[record_set_id])} records.")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"Skipping {record_set_id} due to error: {e}")

# Display the first few rows of the (first) record set
if record_set_ids and record_set_ids[0] in dataframes:
    print(f"\nFirst few rows for record set {record_set_ids[0]}:")
    display(dataframes[record_set_ids[0]].head())
else:
    print("No record sets loaded successfully.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records, normalize a numeric column, and group by a categorical field. All fields referenced by `@id`.


In [ ]:
# Pick the first available record set for EDA
from numpy import number

if record_set_ids and record_set_ids[0] in dataframes and not dataframes[record_set_ids[0]].empty:
    df = dataframes[record_set_ids[0]]
    print(f"Record set @id for EDA: {record_set_ids[0]}")
    # Find numeric columns (by checking dtype or metadata annotation if available)
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    group_fields = df.select_dtypes(exclude=['number']).columns.tolist()
    print(f"Numeric fields: {numeric_cols}")
    print(f"Non-numeric/grouping candidate fields: {group_fields}")

    if numeric_cols:
        # Use the first numeric field for processing
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with '{{}}' > {{:.2f}}: {{}} records".format(numeric_field, threshold, len(filtered_df)))

        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"First rows with normalized {numeric_field}:")
        display(filtered_df[[numeric_field, col_norm]].head())

        # Group by the first available non-numeric field
        group_field = group_fields[0] if group_fields else None
        if group_field and group_field in filtered_df.columns:
            print(f"Grouped mean by '{group_field}':")
            group_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(group_df.head())
        else:
            print("No group field available for grouping.")
    else:
        print("No numeric fields in this record set.")
else:
    print("No data available for EDA in the selected record set.")

## 5. Visualization
Visualize data distributions or field relationships using `matplotlib` and `seaborn` on the first record set (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and record_set_ids[0] in dataframes and not dataframes[record_set_ids[0]].empty:
    df = dataframes[record_set_ids[0]]
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if len(numeric_cols) >= 1:
        field = numeric_cols[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[field], bins=25, kde=True)
        plt.title(f"Distribution of {field}")
        plt.xlabel(field)
        plt.ylabel("Count")
        plt.show()

    if len(numeric_cols) >= 2:
        plt.figure(figsize=(7, 6))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f"Scatter plot of {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.show()
else:
    print("No visualization possible. No data available.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a FAIR-compliant Croissant dataset using the `mlcroissant` library. We:
- Reviewed metadata such as title and description
- Enumerated record sets and fields (referenced by `@id`)
- Loaded example data into Pandas DataFrames
- Performed basic EDA and grouping by field
- Visualized field distributions

Refer to the dataset's documentation and Croissant schema for further details or for in-depth domain-specific analyses.